# 01. Search Papers

OpenAlex API로 키워드 검색 → `data/papers_raw.csv`로 저장.

**OpenAlex**는 무인증·무료·일일 100k 호출이고 모든 분야 + 인용 데이터를 줍니다. 가장 부담 없이 시작할 수 있습니다.

추후 arXiv, Semantic Scholar로 확장 가능 — 같은 노트북 안에 별도 셀로 추가하세요.

In [ ]:
!pip install -r ../../../requirements.txt

In [2]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from typing import Optional

DATA_DIR = Path('../data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 본인 이메일을 넣으면 OpenAlex가 'polite pool'에 넣어 더 안정적인 응답을 줍니다.
MAILTO = os.environ.get('OPENALEX_MAILTO', 'student@example.com')


## 검색 파라미터

`topic_scoping(rr)`에서 받은 키워드와 연도 범위를 여기에 넣습니다.

In [3]:
QUERY = 'agent architectures large language models'   # ← 본인 키워드로 교체
FROM_YEAR = 2020
TO_YEAR = 2026
MAX_RESULTS = 50

## 검색 함수

기본 호출 외에 **선택 옵션**으로 학생이 본인 주제에 정밀도를 맞출 수 있습니다.

| 옵션 | 의미 | 사용 예 |
|---|---|---|
| `search_field='title_and_abstract'` | 제목·abstract만 검색 (저자명·기관 매칭 노이즈 ↓) | 좁은 주제 정밀 검색 |
| `language='en'` | 영어 논문만 | 국제 학술지 위주 |
| `language='ko'` | 한국어 논문만 | 국내 사례·정책 |
| `country_code='KR'` | 저자 기관이 한국 | 국내 연구 동향 |
| `max_retries=5` | 네트워크 재시도 횟수 | 강의실 Wi-Fi 불안정 시 |

기본 호출(아무 옵션 없이)은 가장 넓은 검색 — 첫 시도 권장.

In [4]:
def search_openalex(
    query: str,
    from_year: int,
    to_year: int,
    max_results: int = 50,
    mailto: str = MAILTO,
    *,
    search_field: str = 'default',          # 'default' | 'title' | 'title_and_abstract'
    language: Optional[str] = None,          # 'en', 'ko' 등 ISO-639-1
    country_code: Optional[str] = None,      # 'KR', 'US' 등 (저자 기관 기준)
    max_retries: int = 3,
    retry_backoff: float = 2.0,
) -> pd.DataFrame:
    """OpenAlex Works 검색 — 페이지네이션 + 재시도 + 선택적 필터 포함.

    학생이 자기 주제에 적용할 때 한 줄 옵션으로 정밀도·범위·언어를 조정 가능.
    """
    base_url = 'https://api.openalex.org/works'

    # 1) 검색 정밀도 — search_field에 따라 query를 다른 파라미터로 보냄
    extra_filter = None
    base_params = {'mailto': mailto}
    if search_field == 'title':
        extra_filter = f'title.search:{query}'
    elif search_field == 'title_and_abstract':
        extra_filter = f'title_and_abstract.search:{query}'
    else:
        base_params['search'] = query

    # 2) 필터 조합 (date 필수 + 옵션)
    filters = [
        f'from_publication_date:{from_year}-01-01',
        f'to_publication_date:{to_year}-12-31',
    ]
    if language:
        filters.append(f'language:{language}')
    if country_code:
        filters.append(f'authorships.institutions.country_code:{country_code.lower()}')
    if extra_filter:
        filters.append(extra_filter)
    base_params['filter'] = ','.join(filters)

    # 3) 페이지네이션 + 재시도
    rows, cursor = [], '*'
    while len(rows) < max_results:
        params = {
            **base_params,
            'cursor': cursor,
            'per-page': min(200, max_results - len(rows)),
        }

        for attempt in range(max_retries):
            try:
                r = requests.get(base_url, params=params, timeout=30)
                r.raise_for_status()
                payload = r.json()
                break
            except requests.RequestException as e:
                if attempt == max_retries - 1:
                    raise RuntimeError(
                        f'OpenAlex 요청 실패 ({attempt+1}/{max_retries}): {e}'
                    ) from e
                time.sleep(retry_backoff ** attempt)

        for w in payload.get('results', []):
            rows.append(_to_row(w))
            if len(rows) >= max_results:
                break

        cursor = (payload.get('meta') or {}).get('next_cursor')
        if not cursor:
            break

    return pd.DataFrame(rows)


def _to_row(w: dict) -> dict:
    """OpenAlex Work → 한 행. 저자 'X 외 N명' + citations_per_year 포함."""
    auths = w.get('authorships', []) or []
    head = ', '.join(a['author']['display_name'] for a in auths[:3])
    authors_str = head + (f' 외 {len(auths)-3}명' if len(auths) > 3 else '')

    year = w.get('publication_year')
    cit = w.get('cited_by_count', 0) or 0
    age = max(2026 - (year or 2026), 1)            # 0년 차 방지
    cit_per_year = round(cit / age, 2)

    return {
        'id': w.get('id'),
        'title': w.get('title'),
        'authors': authors_str,
        'year': year,
        'venue': ((w.get('primary_location') or {}).get('source') or {}).get('display_name'),
        'cited_by_count': cit,
        'citations_per_year': cit_per_year,
        'language': w.get('language'),
        'doi': w.get('doi'),
        'oa_url': (w.get('open_access') or {}).get('oa_url'),
        'abstract': _reconstruct_abstract(w.get('abstract_inverted_index')),
    }


def _reconstruct_abstract(inv_index):
    if not inv_index:
        return None
    positions = []
    for word, idxs in inv_index.items():
        for i in idxs:
            positions.append((i, word))
    positions.sort()
    return ' '.join(w for _, w in positions)


df = search_openalex(QUERY, FROM_YEAR, TO_YEAR, MAX_RESULTS)
print(f'{len(df)} papers fetched')
df.head()


50 papers fetched


,id,title,authors,year,venue,cited_by_count,citations_per_year,language,doi,oa_url,abstract
0,https://openalex.org/W4387835442,Generative Agents: Interactive Simulacra of Hu...,"Joon Sung Park, Joseph O’Brien, Carrie J. Cai ...",2023,None,1308,436.0,en,https://doi.org/10.1145/3586183.3606763,https://dl.acm.org/doi/pdf/10.1145/3586183.360...,Believable proxies of human behavior can empow...
1,https://openalex.org/W4390721568,From LLM to Conversational Agent: A Memory Enh...,"Na Liu, Liangyu Chen, Xiaoyu Tian 외 3명",2024,arXiv (Cornell University),6,3.0,en,https://doi.org/10.48550/arxiv.2401.02777,https://arxiv.org/pdf/2401.02777,This paper introduces RAISE (Reasoning and Act...
2,https://openalex.org/W4387323291,Improving Planning with Large Language Models:...,"Taylor W. Webb, Shanka Subhra Mondal, Ida Mome...",2023,arXiv (Cornell University),3,1.0,en,https://doi.org/10.48550/arxiv.2310.00194,https://arxiv.org/pdf/2310.00194,Large language models (LLMs) demonstrate impre...
3,https://openalex.org/W4414432626,A Large Language Model-Enabled Control Archite...,"Jonghan Lim, Ilya Kovalenko",2025,None,4,4.0,en,https://doi.org/10.1109/case58245.2025.11163802,None,Manufacturing environments are becoming more c...
4,https://openalex.org/W4304195432,Persona-Driven Benchmarking for Generalizable ...,Ishan Katoch,2022,arXiv (Cornell University),528,132.0,en,https://doi.org/10.48550/arxiv.2210.03629,https://arxiv.org/pdf/2210.03629,"This research paper, ""Persona-Driven Benchmark..."


## (선택) 본인 주제로 정밀 재검색

기본 검색 결과가 너무 넓거나 노이즈가 많으면 아래 셀의 옵션을 조정해 다시 호출하세요. 같은 `df`로 덮어쓰면 다음 셀에서 그대로 저장됩니다.

In [5]:
# 예시 1 — 제목·abstract만 검색 (노이즈 최소화)
# df = search_openalex(QUERY, FROM_YEAR, TO_YEAR, max_results=100,
#                      search_field='title_and_abstract')

# 예시 2 — 영어 논문만, 200건까지
# df = search_openalex(QUERY, FROM_YEAR, TO_YEAR, max_results=200,
#                      search_field='title_and_abstract', language='en')

# 예시 3 — 한국어 + 한국 기관 저자 (국내 연구 발굴)
# df = search_openalex('국부론 자유시장', 2015, 2026, 50,
#                      language='ko', country_code='KR')

# 위 예시 중 하나의 주석을 풀고 실행하면 df가 업데이트됩니다.
df.head()

,id,title,authors,year,venue,cited_by_count,citations_per_year,language,doi,oa_url,abstract
0,https://openalex.org/W4387835442,Generative Agents: Interactive Simulacra of Hu...,"Joon Sung Park, Joseph O’Brien, Carrie J. Cai ...",2023,None,1308,436.0,en,https://doi.org/10.1145/3586183.3606763,https://dl.acm.org/doi/pdf/10.1145/3586183.360...,Believable proxies of human behavior can empow...
1,https://openalex.org/W4390721568,From LLM to Conversational Agent: A Memory Enh...,"Na Liu, Liangyu Chen, Xiaoyu Tian 외 3명",2024,arXiv (Cornell University),6,3.0,en,https://doi.org/10.48550/arxiv.2401.02777,https://arxiv.org/pdf/2401.02777,This paper introduces RAISE (Reasoning and Act...
2,https://openalex.org/W4387323291,Improving Planning with Large Language Models:...,"Taylor W. Webb, Shanka Subhra Mondal, Ida Mome...",2023,arXiv (Cornell University),3,1.0,en,https://doi.org/10.48550/arxiv.2310.00194,https://arxiv.org/pdf/2310.00194,Large language models (LLMs) demonstrate impre...
3,https://openalex.org/W4414432626,A Large Language Model-Enabled Control Archite...,"Jonghan Lim, Ilya Kovalenko",2025,None,4,4.0,en,https://doi.org/10.1109/case58245.2025.11163802,None,Manufacturing environments are becoming more c...
4,https://openalex.org/W4304195432,Persona-Driven Benchmarking for Generalizable ...,Ishan Katoch,2022,arXiv (Cornell University),528,132.0,en,https://doi.org/10.48550/arxiv.2210.03629,https://arxiv.org/pdf/2210.03629,"This research paper, ""Persona-Driven Benchmark..."


In [6]:
out = DATA_DIR / 'papers_raw.csv'
df.to_csv(out, index=False)
print(f'Saved → {out.resolve()}')

Saved → /Users/sungjae-cha/Documents/06 아름다운서당/next-seodang/projects/02_research_report_helper/data/papers_raw.csv


## 다음 단계

`02_score_quality.ipynb`을 열어 인용수·연도 기반 정량 점수를 계산하세요.